# СИМА — Демонстрация сервиса вычисления рельефаЕдиный демонстрационный ноутбук для `sima-relief-service`.Сервис реализует блок «Анализ рельефа» (Q3-концепция):- Оценка материалов (LAS/TIFF)- Расчёт ЦМР (DTM) и ЦММ (DSM)- Производные: уклон, экспозиция, TPI- Векторные слои: горизонтали, отметки высот, TINПараметры выровнены с легаси-прототипом (ANALYSIS_REPORT.md).Поддерживаются два датасета: **demo** (pt000100.las) и **test** (P-42-041-239, с эталоном).

## 1. Импорт и настройка путей

In [ ]:
import sys, osfrom pathlib import PathBACKEND = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')for pkg in [    'packages/sima-dem-core/src',    'packages/sima-dem-ground/src',    'packages/sima-dem-dsm/src',    'packages/sima-relief-service/src',]:    p = str(BACKEND / pkg)    if p not in sys.path:        sys.path.insert(0, p)import rasterioimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.colors as mcolorsfrom sima_relief_service import (    ReliefService, ReliefRequest, ReliefParams, TileInput,    DerivativesParams, VectorsParams, HeightsParams, SmrfParams,    SmoothingParams, DsmParams, assess_materials,)print('sima-relief-service loaded')

## 2. Выбор датасета- `demo` — pt000100.las (без эталона)- `test` — P-42-041-239-g_ground_TLO.las (с эталонным DSM для сравнения)

In [ ]:
DATASET = 'test'  # 'demo' or 'test'SIMA_ROOT = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА')if DATASET == 'demo':    DATA_DIR = SIMA_ROOT / '23_04_12_digital_elevation_1-46-315/demo_data'    LAS_PATH = str(DATA_DIR / 'pt000100.las')    TIF_PATH = str(DATA_DIR / '00000100.tif')    REFERENCE_DSM = Noneelif DATASET == 'test':    DATA_DIR = SIMA_ROOT / 'test_data'    LAS_PATH = str(DATA_DIR / 'P-42-041-239-g_ground_TLO.las')    TIF_PATH = str(DATA_DIR / 'P-42-041-239-g.tif')    REFERENCE_DSM = str(DATA_DIR / 'P-42-041-239-g_DSM.tif')OUTPUT_DIR = str(BACKEND / 'output_relief_demo' / DATASET)os.makedirs(OUTPUT_DIR, exist_ok=True)RESOLUTION = 1.0crs_source = REFERENCE_DSM if REFERENCE_DSM else TIF_PATHwith rasterio.open(crs_source) as src:    CRS = src.crs.to_wkt()print(f'Dataset: {DATASET}')print(f'LAS:  {LAS_PATH}')print(f'TIF:  {TIF_PATH}')print(f'CRS:  {CRS[:80]}...')print(f'Res:  {RESOLUTION} m')print(f'Ref:  {REFERENCE_DSM}')

## 2b. Восстановление абсолютных Z (для test-датасета)Test-датасет содержит TLO (HeightAboveGround). Нужно восстановить абсолютные Zпо эталонному DSM перед построением ЦМР.

In [ ]:
if REFERENCE_DSM:    sys.path.insert(0, str(BACKEND))    from tests.fixtures.restore_las import restore_absolute_las    restored = str(Path(OUTPUT_DIR) / 'restored_absolute.las')    restore_absolute_las(LAS_PATH, REFERENCE_DSM, restored)    LAS_PATH = restored    print(f'Восстановлено: {LAS_PATH}')else:    print('Восстановление не требуется (demo-датасет)')

## 3. Оценка исходных материалов

In [ ]:
assessment = assess_materials(vls_files=[LAS_PATH], afs_files=[TIF_PATH])print(assessment)

## 4. Параметры расчёта рельефаПараметры выровнены с легаси-прототипом и `run_dsm_demo.py`:- SMRF: slope=0.2, window=16, threshold=0.45, scalar=1.2 (без cut_smrf)- Сглаживание: sigma=2.0 (x resolution)- Интерполяция: IDW, max_search_distance=100, fallback_to_min_z=True- TPI: радиусы 270/810/2430 м- Горизонтали: 0.5, 2.0, 5.0, 10.0 м- Отметки высот: каждая 10-я точка ground (source=las)- ЦММ (DSM): output_type=max

In [ ]:
params = ReliefParams(    target_crs=CRS,    filter_method='smrf',    smrf=SmrfParams(slope=0.2, window=16, threshold=0.45, scalar=1.2),    smoothing=SmoothingParams(enabled=True, sigma=2.0, order=0, window=5),    dsm=DsmParams(enabled=True, output_type='max', interpolate=True, fill_holes=True),    derivatives=DerivativesParams(        slopes=True, slopes_res=RESOLUTION,        aspect=True, aspect_res=RESOLUTION,        tpi=True, tpi_radii=[270, 810, 2430],        interpolation=True, inter_amp=100,    ),    vectors=VectorsParams(horizontals=[0.5, 2.0, 5.0, 10.0], tin=True),    heights=HeightsParams(enabled=True, source='las', step=10),    deterministic=True, seed=42,)print('Параметры настроены')

## 5. Запуск сервиса рельефаReliefService.run() выполняет конвейер по шагам:crop -> filter -> ЦМР (DTM) -> ЦММ (DSM) -> smooth -> slope/aspect/TPI -> contours/TIN -> heights

In [ ]:
request = ReliefRequest(    params=params,    project_id=f'relief_{DATASET}',    resolution=RESOLUTION,    season='summer',    tiles=[TileInput(name=Path(LAS_PATH).stem, vls_path=LAS_PATH, afs_path=TIF_PATH)],)svc = ReliefService(root_dir=OUTPUT_DIR)result = svc.run(request)print(f'Job status: {result.job.status}')print(f'Tiles done:  {result.job.tiles_done}/{result.job.tiles_total}')if result.job.tiles_failed > 0:    print(f'Tiles failed: {result.job.tiles_failed}')    for t in result.job.tiles:        if t.status == 'failed':            print(f'  {t.name}: {t.reason}')

## 6. Результаты — список артефактов

In [ ]:
for tile in result.job.tiles:    if tile.status != 'done':        continue    print(f'\n=== Tile: {tile.name} ===')    for st in tile.steps:        print(f'  step: {st.name:12s}  status: {st.status:8s}  {st.duration_ms}ms')    print(f'\n  Artifacts ({len(tile.output_files)}):')    for art in tile.output_files:        size_kb = (art.size_bytes or 0) / 1024        print(f'    {art.layer:12s}  {art.kind:8s}  {os.path.basename(art.path):40s}  {size_kb:.0f} KB')

## 7. Визуализация ЦМР и ЦММСравнение DTM (ground, IDW) и DSM (все точки, max).

In [ ]:
def read_raster(path):    with rasterio.open(path) as src:        arr = src.read(1).astype(float)        if src.nodata is not None:            arr = np.where(arr == src.nodata, np.nan, arr)    return arrtile = result.job.tiles[0]art_map = {a.layer: a.path for a in tile.output_files}dtm = read_raster(art_map['dtm'])dsm = read_raster(art_map['dsm']) if 'dsm' in art_map else Nonez_min = min(np.nanmin(dtm), np.nanmin(dsm) if dsm is not None else np.inf)z_max = max(np.nanmax(dtm), np.nanmax(dsm) if dsm is not None else -np.inf)terrain_norm = mcolors.Normalize(vmin=z_min, vmax=z_max)fig, axes = plt.subplots(1, 2 if dsm is not None else 1, figsize=(18 if dsm is not None else 8, 7))if dsm is None:    axes = [axes]im = axes[0].imshow(dtm, cmap='terrain', norm=terrain_norm)axes[0].set_title(f'ЦМР (DTM)\nZ={np.nanmin(dtm):.1f}-{np.nanmax(dtm):.1f} м', fontsize=13)plt.colorbar(im, ax=axes[0], shrink=0.6)if dsm is not None:    im = axes[1].imshow(dsm, cmap='terrain', norm=terrain_norm)    axes[1].set_title(f'ЦММ (DSM)\nZ={np.nanmin(dsm):.1f}-{np.nanmax(dsm):.1f} м', fontsize=13)    plt.colorbar(im, ax=axes[1], shrink=0.6)plt.suptitle(f'СИМА — {DATASET}', fontsize=16)plt.tight_layout()plt.savefig(os.path.join(OUTPUT_DIR, 'dtm_dsm_comparison.png'), dpi=150)plt.show()print(f'DTM: Z={np.nanmin(dtm):.2f}-{np.nanmax(dtm):.2f} м, mean={np.nanmean(dtm):.2f}, std={np.nanstd(dtm):.2f}')if dsm is not None:    print(f'DSM: Z={np.nanmin(dsm):.2f}-{np.nanmax(dsm):.2f} м, mean={np.nanmean(dsm):.2f}, std={np.nanstd(dsm):.2f}')

## 8. Визуализация производных (уклон, экспозиция, TPI)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))for ax, key, title, cmap in [    (axes[0], 'slope', 'Уклон (град)', 'hot'),    (axes[1], 'aspect', 'Экспозиция (град)', 'hsv'),    (axes[2], 'tpi', 'TPI', 'RdBu_r'),]:    if key in art_map:        arr = read_raster(art_map[key])        im = ax.imshow(arr, cmap=cmap)        ax.set_title(title, fontsize=13)        plt.colorbar(im, ax=ax, shrink=0.6)plt.tight_layout()plt.savefig(os.path.join(OUTPUT_DIR, 'derivatives.png'), dpi=150)plt.show()

## 8b. Сравнение методов сглаживания ЦМРСравнение гауссова сглаживания при разных sigma:- sigma=1.0 — слабое сглаживание- sigma=2.0 — стандартное (соответствует легаси)- sigma=4.0 — сильное сглаживаниеТакже сравнение с медианным фильтром.

In [ ]:
from sima_dem_core.raster.smooth import gauss_smoothfrom sima_dem_core.raster.median import med_filterimport shutildtm_path = art_map['dtm']stem = Path(dtm_path).stem.replace('_ground_dem', '')smooth_dir = str(Path(OUTPUT_DIR) / 'smoothing_compare')os.makedirs(smooth_dir, exist_ok=True)smooth_results = {}# Gaussian sigma=1.0p1 = str(Path(smooth_dir) / f'{stem}_gauss_s1.tif')gauss_smooth(dtm_path, p1, sigma=1.0 * RESOLUTION, order=0, window_size=5,             fill_holes=True, max_search_distance=100)smooth_results['Gaussian sigma=1.0'] = p1# Gaussian sigma=2.0 (standard)p2 = str(Path(smooth_dir) / f'{stem}_gauss_s2.tif')gauss_smooth(dtm_path, p2, sigma=2.0 * RESOLUTION, order=0, window_size=5,             fill_holes=True, max_search_distance=100)smooth_results['Gaussian sigma=2.0'] = p2# Gaussian sigma=4.0p4 = str(Path(smooth_dir) / f'{stem}_gauss_s4.tif')gauss_smooth(dtm_path, p4, sigma=4.0 * RESOLUTION, order=0, window_size=5,             fill_holes=True, max_search_distance=100)smooth_results['Gaussian sigma=4.0'] = p4# Median filter (window=5)pm = str(Path(smooth_dir) / f'{stem}_median_w5.tif')shutil.copy2(dtm_path, pm)med_filter(pm, 5)smooth_results['Median window=5'] = pmprint('Сглаживание готово:', list(smooth_results.keys()))

In [ ]:
dtm_arr = read_raster(dtm_path)smooth_arrays = {label: read_raster(p) for label, p in smooth_results.items()}z_min_s = min(np.nanmin(dtm_arr), min(np.nanmin(a) for a in smooth_arrays.values()))z_max_s = max(np.nanmax(dtm_arr), max(np.nanmax(a) for a in smooth_arrays.values()))s_norm = mcolors.Normalize(vmin=z_min_s, vmax=z_max_s)fig, axes = plt.subplots(2, 3, figsize=(21, 12))im = axes[0, 0].imshow(dtm_arr, cmap='terrain', norm=s_norm)axes[0, 0].set_title('ЦМР (без сглаживания)', fontsize=13)plt.colorbar(im, ax=axes[0, 0], shrink=0.6)for ax, (label, arr) in zip(axes.flatten()[1:], smooth_arrays.items()):    diff = np.abs(arr - dtm_arr)    valid = np.isfinite(dtm_arr) & np.isfinite(arr)    rmse = np.sqrt(np.nanmean(diff[valid]**2)) if valid.any() else 0    im = ax.imshow(arr, cmap='terrain', norm=s_norm)    ax.set_title(f'{label}\nRMSE vs DTM={rmse:.3f} м', fontsize=13)    plt.colorbar(im, ax=ax, shrink=0.6)plt.suptitle(f'Сравнение методов сглаживания — {DATASET}', fontsize=16)plt.tight_layout()plt.savefig(os.path.join(OUTPUT_DIR, 'smoothing_comparison.png'), dpi=150)plt.show()print('Сравнение сохранено')

In [ ]:
import pandas as pdstats = []for label, arr in [('DTM (raw)', dtm_arr)] + list(smooth_arrays.items()):    valid = arr[np.isfinite(arr)]    stats.append({        'Метод': label,        'Z min': f'{np.min(valid):.2f}',        'Z max': f'{np.max(valid):.2f}',        'Z mean': f'{np.mean(valid):.2f}',        'Z std': f'{np.std(valid):.4f}',        'Valid %': f'{100*len(valid)/arr.size:.1f}%',    })pd.DataFrame(stats)

## 9. Сглаженная ЦМР

In [ ]:
if 'dtm_smooth' in art_map:    smoothed = read_raster(art_map['dtm_smooth'])    fig, axes = plt.subplots(1, 2, figsize=(18, 7))    s_norm = mcolors.Normalize(vmin=min(np.nanmin(dtm), np.nanmin(smoothed)),                                vmax=max(np.nanmax(dtm), np.nanmax(smoothed)))    for ax, arr, title in [(axes[0], dtm, 'ЦМР (сырая)'), (axes[1], smoothed, 'ЦМР (сглаженная, sigma=2.0)')]:        im = ax.imshow(arr, cmap='terrain', norm=s_norm)        ax.set_title(title, fontsize=13)        plt.colorbar(im, ax=ax, shrink=0.6)    plt.tight_layout()    plt.savefig(os.path.join(OUTPUT_DIR, 'smooth_comparison.png'), dpi=150)    plt.show()    print(f'Smoothed: Z={np.nanmin(smoothed):.2f}-{np.nanmax(smoothed):.2f} м, std={np.nanstd(smoothed):.2f}')else:    print('Сглаживание отключено')

## 10. Сравнение с эталоном (для test-датасета)Требование: RMSE < 5% от среднего Z эталона.

In [ ]:
if REFERENCE_DSM and os.path.exists(REFERENCE_DSM):    ref = read_raster(REFERENCE_DSM)    min_h = min(dtm.shape[0], ref.shape[0])    min_w = min(dtm.shape[1], ref.shape[1])    b, r = dtm[:min_h, :min_w], ref[:min_h, :min_w]    valid = np.isfinite(b) & np.isfinite(r)    diff = np.where(valid, np.abs(b - r), np.nan)    rmse = float(np.sqrt(np.nanmean(diff[valid]**2))) if valid.any() else 0    mean_ref = float(np.nanmean(r[valid])) if valid.any() else 0    rel_err = rmse / mean_ref if mean_ref else 0    print(f'RMSE: {rmse:.4f} м, Mean эталона: {mean_ref:.4f} м')    print(f'Относительная ошибка: {rel_err:.4%}')    print(f'Требование < 5%: {"\u2713 ПРОЙДЕН" if rel_err < 0.05 else "\u2717 НЕ ПРОЙДЕН"}')    fig, axes = plt.subplots(1, 3, figsize=(21, 6))    for ax, title, data in [(axes[0], 'Построено', b), (axes[1], 'Эталон', r)]:        im = ax.imshow(data, cmap='terrain', norm=terrain_norm)        ax.set_title(title, fontsize=13)        plt.colorbar(im, ax=ax, shrink=0.7)    im = axes[2].imshow(diff, cmap='Reds')    axes[2].set_title(f'Разница (RMSE={rmse:.2f} м)', fontsize=13)    plt.colorbar(im, ax=axes[2], shrink=0.7)    plt.tight_layout()    plt.savefig(os.path.join(OUTPUT_DIR, 'reference_comparison.png'), dpi=150)    plt.show()else:    print('Сравнение с эталоном недоступно (demo-датасет без эталона)')

## 11. ИтогСервис `sima-relief-service` производит все артефакты блока «Анализ рельефа» (Q3):| Артефакт | Формат | Q3 ||---|---|---|| ЦМР (.geotiff) | GeoTIFF (IDW) | ✅ || ЦММ (.geotiff) | GeoTIFF (max) | ✅ || Карта уклонов (.geotiff) | GeoTIFF | ✅ || Карта экспозиции (.geotiff) | GeoTIFF | ✅ || TPI (.geotiff) | GeoTIFF | ✅ || Отметки высот (.shp) | Shapefile | ✅ || Горизонтали (.shp) | Shapefile (0.5/2/5/10 м) | ✅ || TIN (.dxf) | DXF | ✅ |